In [ ]:
## ok i am given a problem, that asks me
# you have the ppo rollouts till a timestep, because ppo rollout buffer can only store this much..
# but the environment doesn't terminate it goes, but as of our ppo rollout limit, we don't have their reward etc..
# so, we will use our critic to get the esitmate of the next state...
# and we will use that esitmate to update our rest of the states (backward) ->
# and here, I am assuing i am already given a critic value(that we got from the critic network)

# this is the detail given to us
rewards = [1.0, 2.0, 3.0]
dones = [False, False, False] ## i can see the last value is false, that means the enviroment goes beyond..
gamma = 0.9
last_value = 10.0


def calculate_gt(rewards, dones, gamma, last_value):
    # ok let's initialize our rewards i.e len(rewards) with 0's 
    Gt = [0.0] * len(rewards)  ## initializing empty zero's of len(rewards)
    running_return = last_value ## this is the value we got from the critic

    for t in list(range(len(rewards)-1, -1, -1)):
        if dones[t]:  ## checking if during backward loop --> we encountered the terminal state ---> then our running_return = will be 0.0
            running_return = 0.0
        running_return = rewards[t] + gamma * running_return  ## here for the first time we will use the --> critic value for recursive Gt calculation 
        Gt[t] = running_return  ## now we are updating the Gt list ---> with the exact t with it's corresponded running_return  (i.e Actual Return..)

    return Gt  ## returning here from the function.. 


print(calculate_gt(rewards=rewards,dones=dones, gamma=gamma, last_value=10))



[12.520000000000001, 12.8, 12.0]


#### Generalized Advantage Estimation(GAE)


1. INPUTS

rewards
values
dones
gamma
lambda
last_value


2. OUTPUT

advantage estimate for every timestep


3. STATE TO REMEMBER

Loop 1:
td_residuals list

Loop 2:
one running_advantage variable


4. REPEATED OPERATIONS

Loop 1:
select next_value
compute TD residual
store TD residual

Loop 2:
iterate backward
cut future propagation at terminal state
compute current GAE advantage
store advantage


5. DATA STRUCTURES

td_residuals list
advantages list
one running_advantage variable

In [ ]:
# inputs given
rewards = [1.0, 2.0, 3.0]
values = [5.0, 6.0, 7.0]
dones = [False, False, False]

gamma = 0.9
lam = 0.95
last_value = 10.0



def calculate_advantages(
    rewards,
    values,
    dones,
    gamma,
    lam,
    last_value,
):
    td_residual = [0.0] * len(rewards)
    advantages = [0.0] * len(rewards)
    running_advantage = 0.0

    # Loop 1 --> calculate TD residuals
    for t in range(len(rewards) - 1, -1, -1):

        mask = 1.0 - float(dones[t])

        if dones[t]:
            next_value = 0.0

        elif t == len(rewards) - 1:
            next_value = last_value

        else:
            next_value = values[t + 1]

        residual = (rewards[t] + gamma * mask * next_value - values[t])

        td_residual[t] = residual

    # Loop 2 --> calculate GAE
    for t in range(len(rewards) - 1, -1, -1):

        mask = 1.0 - float(dones[t])

        if dones[t]:
            running_advantage = 0.0

        running_advantage = (td_residual[t] + gamma * lam * mask * running_advantage)

        advantages[t] = running_advantage

    return advantages


print(
    calculate_advantages(
        rewards,
        values,
        dones,
        gamma,
        lam,
        last_value,
    )
)

In [ ]:
## ok, here we are going to implement the proximal policy optimization clipped loss
import torch



def ppo_clipped_loss(old_log_probs, new_log_probs, advantages, clip_eps):
     

    ## here first we will calculate the probability_ratio (as advantages are already given..)
    ratio = torch.exp(new_log_probs - old_log_probs)

    ## now  let's calcualte the unclipped surrogate loss
    unclipped_surr = ratio * advantages

    ## now calculate the clipped ratio ---> this will take probability_ratio --> and clip that..
    clipped_ratio = torch.clamp(ratio, 1-clip_eps, 1+clip_eps)


    ##  now calculate the clipped_surrogate
    clipped_surr = clipped_ratio * advantages


    ### now calculate the objective ...
    objective = torch.min(unclipped_surr, clipped_surr)

    ## calculate the loss
    loss = - (torch.mean(objective))
    

    return loss

In [59]:
"""I am given a problem to implement causal attention masking for financial market sequences.

The raw market input can have shape `[B, T, F]`, where `B` is the batch size, `T` is the number of market time steps, and `F` is the number of numerical market features at each time step.

Before attention, the market features can be projected from the raw feature dimension `F` into the Transformer representation dimension `D` using a learned linear projection:

`[B, T, F] -> [B, T, D]`

The resulting representations can then be passed through the query, key, and value projections. Computing `Q @ K^T` produces attention scores over pairs of time steps, with shape `[B, T, T]` when ignoring attention heads.

For causal masking, I need to prevent each time step from attending to future market time steps. If `i` is the current query time step and `j` is the key time step being attended to, then `j > i` means that `j` is in the future relative to `i`.

Therefore, I will construct a mask of shape `[T, T]`. Positions where `j <= i` are allowed and receive a value of `0`, while positions where `j > i` are future positions and receive `-inf`.

The mask is added to the attention scores before softmax. Allowed scores remain unchanged because `score + 0 = score`, while future scores become `-inf`. After softmax, those future positions receive zero attention probability.

The `[T, T]` causal mask can be broadcast across the batch dimension and, in multi-head attention, across the attention-head dimension.
"""



## ok let's now implement the causal mask for this problem

def causal_masking(seq_len):  ## ok this is the function, that  will take the seq_len as  input and give us the T,T shape causal mask  ....

    matrix = []  ## ok here this is the  empty matrix i am defining, we will store all rows here one by  one ... 

    for i in range(seq_len):  ## here i will iterate row-wise --> first it will take  the row index 1 and write all the columsn associateted with it..after that will write the same in the next row 
        row = [] ## here i am defining the  empty row = list ... this will store al lthe rows,, thhhat the j will appent later.... and then we will append this row into the big matrix  one by one..
        for j in range(seq_len):
            if j <=i : 
                row.append("0.0")
            else: 
                row.append("-inf")
        matrix.append(row)

    return matrix 




In [2]:
import torch

x = torch.tensor([
    [
        [
            [0.02, 100.0, 0.04, 65.0],
            [0.01,  80.0, 0.03, 55.0],
            [0.05,  60.0, 0.08, 72.0],
            [0.03,  40.0, 0.05, 61.0],
            [-0.01, 90.0, 0.02, 48.0],
        ],
        [
            [0.03, 110.0, 0.05, 68.0],
            [-0.01, 90.0, 0.04, 50.0],
            [0.02,  70.0, 0.06, 66.0],
            [0.01,  45.0, 0.04, 59.0],
            [0.02,  95.0, 0.03, 52.0],
        ],
        [
            [-0.02, 105.0, 0.06, 60.0],
            [0.04,  95.0, 0.05, 63.0],
            [0.01,  75.0, 0.07, 64.0],
            [-0.01, 50.0, 0.05, 56.0],
            [0.03, 100.0, 0.04, 58.0],
        ],
    ],

    [
        [
            [0.01, 120.0, 0.03, 62.0],
            [0.02,  85.0, 0.04, 57.0],
            [0.04,  65.0, 0.07, 70.0],
            [0.02,  42.0, 0.05, 60.0],
            [0.00,  88.0, 0.02, 49.0],
        ],
        [
            [0.02, 125.0, 0.04, 66.0],
            [0.00,  92.0, 0.05, 53.0],
            [0.03,  72.0, 0.08, 69.0],
            [0.02,  47.0, 0.04, 62.0],
            [0.01,  97.0, 0.03, 51.0],
        ],
        [
            [-0.01, 115.0, 0.05, 61.0],
            [0.03,  98.0, 0.04, 65.0],
            [0.02,  78.0, 0.06, 67.0],
            [0.00,  52.0, 0.05, 58.0],
            [0.04, 102.0, 0.04, 60.0],
        ],
    ],
], dtype=torch.float32)

In [ ]:
import torch.nn as nn

## ok first we have the x.shape as [b, t, A, F] and we have to get a linear projection  
# i.e we need this shape [b, t, A, D]..


# getting the feature projection
b, t, A, F = x.shape
vocab_size = t


## defining the project using nn.linear layer 
feature_projection = nn.Linear(F, 8) # ---> this will give us the feature projection.. later.
feature_embeddings = feature_projection(x)
#print(feature_embeddings.shape)

## now we have to create the asset embedding
asset_ids = torch.arange(A)

asset_embeddings = nn.Embedding(A, 8)

all_asset_ids = asset_embeddings(asset_ids) ## ok here this will be the shape of asset_embeddings for all the asset ids.. (in this case 5)
print(all_asset_ids.shape)

## now let's get the time embedding and later we will combine this all together and broadcast them to base shape

time_steps = torch.arange(t)

time_step_embeddings = nn.Embedding(t, 8)

all_time_step_embeddings = time_step_embeddings(time_steps)  ## ok this is our embedding for all the time steps...we will use it for all assets embedding
print(f"Time embedding shape:{all_time_step_embeddings.shape}")

##  now combine...
# first we will add the feature embedding and asset embeddings..

# the feature embedding shape is [b,t,A,D]
# the asset embedding shape is  [A,D]---> so we have to add unsqueeze it and add the single dimension
all_asset_embeddings = all_asset_ids.unsqueeze(0).unsqueeze(0)
print(all_asset_embeddings.shape)  ##  ok now we got the shape of [1,1,5,8], so we can broadcast properly...

first_combined = torch.add(feature_embeddings, all_asset_embeddings)
print(first_combined.shape)

## ok now we have to do the similar operation with time embedding, so lets give it the proper shape...by using unsqueeze to add the dimension where it needs
# first let's unsqueeze it...
all_time_step_embeddings = all_time_step_embeddings.unsqueeze(0).unsqueeze(2)
print(f"Time embedding shape:{all_time_step_embeddings.shape}")

## now last addition operation

x_new = torch.add(first_combined, all_time_step_embeddings)

print(f"Final shape after all embedding operations:{x_new.shape}")


## flatten operation
final_embeddings = x_new.reshape(b, t*A, 8)









torch.Size([5, 8])
Time embedding shape:torch.Size([3, 8])
torch.Size([1, 1, 5, 8])
torch.Size([2, 3, 5, 8])
Time embedding shape:torch.Size([1, 3, 1, 8])
Final shape after all embedding operations:torch.Size([2, 3, 5, 8])


In [20]:
### anagram DSA problem.

inputs = ["eat", "tea", "tan", "ate", "nat", "bat"]
outputs = [["eat", "tea", "ate"], ["tan", "nat"], ["bat"]]


##  ok i have been given a list of words, and i have to find anagrams'like the words containing same characters---> shuffled  here and there..

##  we will use the character counting technique,,--> like will design a vector map of alphabets from a-z ...
## whereevery like we have the character for each word we will increament that..

## [a-z only with zeros ...]
## and when we will loop on each text... and then inside each character... we  will see when  we will encounter word  we will  increament the index value of that character in vector [a-z]

##   first let's do one  thing ...mappping --> a=0, b=1, c=2 etc.. till 25 for 26  alphabetss..

char_mapping = {}

## ascii mapping...-> 97 to a and 123 to z
import string
alphabets = string.ascii_lowercase

for i in range(97, 123):
    index = i - 97
    char_mapping[alphabets[index]] = index




inputs = ["eat", "tea", "tan", "ate", "nat", "bat"]

## 

hashmap = {}  ## hashmap


for text in inputs: ## loop over text
    frequency = [0] * 26
    for char in text: ## loop over char inside each  text pair
        index = char_mapping[char]
        frequency[index] += 1

    sig = tuple(frequency)

    if sig in hashmap:
        hashmap[sig].append(text)
    else:  
        hashmap[sig] = [text]




output = list(hashmap.values())
print(output)






[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


Given an integer array nums, return an array answer such that answer[i] is equal to the product of all the elements of nums except nums[i].

The product of any prefix or suffix of nums is guaranteed to fit in a 32-bit integer.

You must write an algorithm that runs in O(n) time and without using the division operation.

 

Example 1:

Input: nums = [1,2,3,4]
Output: [24,12,8,6]

In [ ]:





inputs = [1,2,3,4]


# we will use the two loops they will run in linear time and  only use  linear space..

# left loop
answer = [1] * len(inputs)
last_value = 1

for i in range(len(inputs)):
    answer[i] = last_value
    last_value *= inputs[i]

print(answer) ## this will run linear time 0(n) --

last_value = 1
for i in range(len(inputs) -1, -1, -1): ## it will run from -> [3,2,1,0]    

    answer[i] *= last_value

    last_value *= inputs[i] 

    ## this  will also run linear time with linear space...

print(answer)






## done this will run  linear time...  in both time and space complexity..

[1, 1, 2, 6]
[24, 12, 8, 6]


3Sum

Given an integer array nums, return all the unique triplets [nums[i], nums[j], nums[k]] such that:

i, j, and k are different indices.
nums[i] + nums[j] + nums[k] == 0.

The solution set must not contain duplicate triplets.

Example 1

Input

nums = [-1,0,1,2,-1,-4]

Output

[[-1,-1,2],[-1,0,1]]

In [ ]:
nums = [-1,0,1,2,-1,-4]


## ook here i have to implement  the 3sum problem, that says find the tiplets  whose sum  == 0, the three elements should be unique,, and it should run in a linear time..


n = len(nums)
nums = sorted(nums)  #3 sort the numbers first 
result = []   # store the triplets.

# loop here
for i in range(n-2):  ## last two pointer will take care of the lsat two numbers in array.

    if i > 0 and nums[i] == nums[i - 1]:   ## if there is something duplicate, move to the next iteration
        continue

    left = i + 1
    right = n - 1

    while left < right:
        total = nums[i] + nums[left] + nums[right]

        if total == 0:
            result.append([nums[i], nums[left],  nums[right]])

            left += 1
            right -= 1

            while left < right and nums[left] == nums[left - 1]:  ## check the duplidate from left
                left += 1   # increament it


            while left < right and nums[right] == nums[right + 1]:  ## check the duplidate from right
                right -= 1



        elif total < 0:
            left += 1

        else: 
            right -= 1




print(result)

        


[[-1, -1, 2], [-1, 0, 1]]


Input:
s = "abcabcbb"

Output:
3

Explanation:
The longest substring is "abc".

In [9]:
s = "abcabcbb"
n = len(s)

seen = set()
left = 0
max_length = 0

for right in range(n):

    while s[right] in seen:
        seen.remove(s[left])
        left += 1

    seen.add(s[right])

    current_length = right - left + 1
    max_length = max(max_length, current_length)

print(max_length)

3


Problem: Container With Most Water

You are given an integer array height of length n.

Each height[i] represents the height of a vertical line drawn at index i.

Find two lines that together with the x-axis form a container that stores the maximum amount of water.

Return the maximum amount of water.

Example
Input:
height = [1,8,6,2,5,4,8,3,7]

Output:
49

In [18]:
## ok let's do this problem again..



## optimzied solution runs linear time---> 0(n)


height = [1,8,6,2,5,4,8,3,7]



## here here i have to find the two numbers in this array in a way that it can maximize the total area..

n = len(height)
max_area = 0

left = 0
right = n-1


while left < right:

    width = right - left
    min_wall = min(height[left], height[right])

    # compute the area first
    area = width * min_wall

    max_area = max(max_area, area)

    if height[left] < height[right]:
        left += 1

    else:
        right -= 1

print(max_area)

49


Given an integer array nums, return all the triplets [nums[i], nums[j], nums[k]] such that i != j, i != k, and j != k, and nums[i] + nums[j] + nums[k] == 0.

Notice that the solution set must not contain duplicate triplets.

 

Example 1:

Input: nums = [-1,0,1,2,-1,-4]
Output: [[-1,-1,2],[-1,0,1]]

In [ ]:
## 3Sum problem ---> (two pointer approach)

nums = [-1,0,1,2,-1,-4]

## ok i have been given a problem to return triplets from the given array, so that their sum == 0. and solution must not contain the duplicate triplets only unique triplets ..ok

## so i will approach this problem first using(brute force approach)

## that will take use of 3 loops and the time complexity will be 0(n^3)
nums = [-1,0,1,2,-1,-4]
n = len(nums)

seen = set()
result = []

for i in range(n-2):
    for j in range(i+1, n-1):
        for k in range(j+1, n):

            total = nums[i] + nums[j] + nums[k]

            if total == 0:
                triplet = tuple(sorted([nums[i], nums[j], nums[k]]))
                if triplet not in seen:
                    seen.add(triplet)
                    result.append(list(triplet))
            



print(result)






[[-1, 0, 1], [-1, -1, 2]]


In [41]:
## ok let's implement the optimized solution now

nums = [-1,0,1,2,-1,-4]

n = len(nums)
nums = sorted(nums)
result = []

for i in range(n-2):

    left = i + 1
    right = n - 1

    ## check the duplicate first (the number we are testing right now--shouldn't be duplicate--that means it's already computed) ## here we are  moving step by step with "i"
    if i > 0 and nums[i] == nums[i-1]:
        continue    ## this means we we are currently on a number, that we have are already searched  the sum == 0 for..

    ## now go from left to right or increase decrease left, right to find 3 numbers and if there sum is equal 0. append them to list
    while left < right:
        total = nums[i] + nums[left] + nums[right]

        if total == 0:
            result.append([nums[i], nums[left], nums[right]])

            ## moving the left and right pointer
            left += 1
            right -= 1

            while left < right and nums[left] == nums[left - 1]:
                left += 1

            while left < right and nums[right] == nums[right + 1]:
                right -= 1

        elif total < 0:
            left += 1

        else:
            right -= 1


print(result)





[[-1, -1, 2], [-1, 0, 1]]


Given a sorted array that has been rotated at some pivot, and a target value, return its index. If the target is not present, return -1.

In [ ]:
nums = [4,5,6,7,0,1,2]
target = 0

## ok i have been given a problem to return and index of target, in an rotated sorted array.

## i will use the binary search, but i have to take care of the sorted and unsorted path, cause there will be a pivot in the array, that will seperate the array into two halves (one as sorted and another as un-sorted.)


n = len(nums)

left = 0
right = n - 1
while left <= right:

    mid =  (left + right) // 2

    if nums[mid] == target: ## if 
        print(mid)
        break


    ## check if the left halve is sorted
    if nums[left] <= nums[mid]: ##  in the left halve
        if nums[left] <= target < nums[mid]:
            right = mid - 1
        else:
            left = mid + 1

    else: 
        if nums[mid] < target <= nums[right]:
            left = mid + 1
        else:
            right = mid - 1


else:
    print(-1)
        



4


In [ ]:
## ok let's solve the islands problem using dfs

class Islands:
    def solution(self, grid):

        cols = len(grid[0])
        rows = len(grid)

        has_visited = set()

        islands = 0

        def dfs(r,c):

            if r < 0 or r >= rows or c < 0 or c >= cols:
                return  

            if (r,c) in has_visited:
                return 

            if grid[r][c] == 0:
                return 

            has_visited.add((r,c))

            dfs(r+1, c)  ## down
            dfs(r-1, c)   ## up
            dfs(r, c-1)  ## left
            dfs(r, c+1)   ## right 

        for r in range(rows):
            for c in range(cols):
                if (r,c) not in has_visited and grid[r][c] == 1:
                    islands += 1
                    dfs(r,c)

        return islands


grid = [[1,1,0,0,0],
        [1,1,0,0,0],
        [0,0,1,0,0],
        [0,0,0,1,1]]

result = Islands().solution(grid=grid)
print(result)


0


In [3]:
grid = [
        ["1","1","0","0","0"],
        ["1","1","0","0","0"],
        ["0","0","1","0","0"],
        ["0","0","0","1","1"]
        ]

print(len(grid[0]))
grid[0]
print(len(grid))

5
4
